# Beckhoff CX2040 / TwinCAT 3 — Compressor Rig PLC Design

**Axial compressor rig:** D_tip = 900 mm · N_op = 3 500 RPM · P_shaft = 320 kW  
**VFD:** ABB ACS880 355 kW + FECA-01 EtherCAT adapter  
**Safety:** TwinSAFE EL6910 · SIL 2 / PL d · IEC 62061

---

This notebook documents and simulates the PLC control system for the rotating rig.  
It is structured as a design reference that can also be executed to verify logic,  
generate the I/O list, simulate the state machine, and produce the throttle sweep schedule.

**Sections**
1. Hardware platform & EtherCAT slot allocation  
2. Imports & dependencies  
3. Setpoints — all derived from the rig design modules  
4. I/O list — ISA-5.1 tagged, Beckhoff terminal mapped  
5. State machine — 9 states, strict transition enforcement  
6. Interlock logic — TwinSAFE safety layer  
7. Pre-start checklist  
8. Speed ramp controller (ACS880 CSV mode)  
9. Throttle sweep controller  
10. Demonstrations & verification  
11. TwinCAT ADS / Python research interface

---
## 1. Hardware Platform & EtherCAT Slot Allocation

### Controller
| Parameter | Value |
|---|---|
| CPU | Beckhoff CX2040 — Intel Core i7 quad-core, 1.91 GHz |
| RAM / Storage | 4 GB RAM · 8 GB CFast |
| OS | Windows 10 IoT Enterprise + TwinCAT 3 runtime |
| EtherCAT | Master built-in (no add-on card needed) |
| IDE | TwinCAT 3 XAE (Visual Studio shell) |

### TwinCAT 3 Task Structure
| Task | Cycle | Core | Function |
|---|---|---|---|
| `SAFE_TASK` | 10 ms | Core 2 (isolated) | TwinSAFE — interlocks, E-stop, STO |
| `MOTION_TASK` | 1 ms | Core 3 (isolated) | EtherCAT I/O scan, ACS880 CSV velocity update |
| `CTRL_TASK` | 5 ms | Core 1 | State machine, PIDs, alarm evaluation |
| `HMI_TASK` | 100 ms | Core 0 | HMI server, OPC-UA TF6100, ADS endpoint |

### VFD
| Parameter | Value |
|---|---|
| Drive | ABB ACS880-01-385A-3, 355 kW, 400 V |
| Fieldbus adapter | FECA-01 EtherCAT adapter (option slot D) |
| Control profile | CiA 402 DS402 — Cyclic Synchronous Velocity (CSV) |
| Update rate | 1 ms (EtherCAT cycle) |
| STO | IEC 61800-5-2 STO via EL2904 TwinSAFE DO |

### EtherCAT I/O Slot Allocation
| Slot | Terminal | Channels | Function |
|---|---|---|---|
| 1 | EK1100 | — | EtherCAT coupler — DIN rail head |
| 2 | EL1904 | 4× safe DI | TwinSAFE — E-stops ×3 + overspeed relay |
| 3 | EL1904 | 4× safe DI | TwinSAFE — door, guard, motor relay, STO-FB |
| 4 | EL2904 | 4× safe DO | TwinSAFE — K1 contactor, ACS880 STO, spare ×2 |
| 5 | EL1252 | 2× fast DI | 1 µs timestamp — ST-101 OPR tachoprobe |
| 6 | EL1008 | 8× DI | VFD status, key switch, pushbuttons |
| 7 | EL2008 | 8× DO | Beacons, hooter, DAQ triggers |
| 8 | EL3174 | 4× AI | 4-20 mA — PT-101, PT-102, PT-201, TT-101 |
| 9 | EL3174 | 4× AI | 4-20 mA — ZT-401, VT-201, VT-202, spare |
| 10 | EL3202 | 2× RTD | PT100 — TE-201 bearing A, TE-202 bearing B |
| 11 | EL3202 | 2× RTD | PT100 — TE-301 motor winding, spare |
| 12 | EL4004 | 4× AO | 4-20 mA — ZC-401 throttle setpoint, spare ×3 |
| 13 | EL9011 | — | EtherCAT bus end terminal |

> **EL1252 note:** At 3 500 RPM the OPR pulse period is ~17 ms. The EL1252 timestamps edges at 1 µs, enabling accurate speed measurement from a single-tooth tachoprobe. A standard EL1008 (3 ms filter) would work but gives lower accuracy; EL1252 is preferred.

---
## 2. Imports & Dependencies

In [ ]:
from dataclasses import dataclass, field
from enum import Enum, auto
from typing import Optional
import time
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Notebook display helpers
from IPython.display import display
try:
    import pandas as pd
    HAS_PANDAS = True
except ImportError:
    HAS_PANDAS = False
    print("pandas not found — tabular output will use plain text")

print("Imports OK")

---
## 3. Setpoints

All setpoints are derived from the aerodynamic and mechanical design modules.  
Change values here; all downstream cells reference this single `RigSetpoints` instance.

| Setpoint | Value | Source |
|---|---|---|
| N_op | 3 500 RPM → 118.2 Hz VFD | Meanline design point |
| N_sw_trip | 3 850 RPM (110%) | Software — TwinSAFE |
| N_hw_trip | 4 000 RPM (114%) | Hardwired EL1904 + relay |
| Skip band 1 | 2 191–2 591 RPM | 2× EO crossing at 2 391 RPM ±200 RPM |
| T_bearing trip | 100 °C | SKF 7211 BEP grease limit |
| Vibration trip | 7.1 mm/s RMS | ISO 10816-3 Zone C/D |
| Throttle range | 30–95% | Aerodynamic φ sweep limits |

In TwinCAT 3 these are stored as a `STRUCT` in a Global Variable List (GVL),  
modifiable from the HMI with password protection (level 2).

In [ ]:
@dataclass
class RigSetpoints:
    """
    All rig process setpoints — derived from the aerodynamic and mechanical
    design modules.  Modify here; state machine and interlock logic reference
    this single instance.

    TwinCAT implementation:
      Stored as STRUCT RigSetpoints in GVL_Settings.
      HMI-editable with level-2 password.
      Change log written to TwinCAT EventLogger on each write.
    """
    # Speed [RPM]
    N_idle_RPM:             float = 0.0
    N_min_stable_RPM:       float = 500.0    # minimum for bearing oil film
    N_op_RPM:               float = 3500.0   # design operating point
    N_sw_trip_RPM:          float = 3850.0   # 110% — software trip
    N_hw_trip_RPM:          float = 4000.0   # 114% — hardwired relay

    # Campbell skip bands [RPM]
    # 2× engine order crosses first critical (4 782 RPM) at 2 391 RPM
    skip1_lo_RPM:           float = 2191.0
    skip1_hi_RPM:           float = 2591.0
    skip_ramp_RPM_s:        float = 80.0     # minimum traverse rate [RPM/s]

    # VFD / motor (4-pole, 50 Hz, nominal 1 480 RPM)
    motor_nominal_RPM:      float = 1480.0
    motor_nominal_Hz:       float = 50.0
    vfd_max_Hz:             float = 120.0
    ramp_up_s:              float = 60.0     # 0 → N_op
    ramp_dn_s:              float = 45.0     # N_op → 0
    bearing_warmup_s:       float = 30.0     # dwell at N_min_stable

    # Bearing temperatures — SKF 7211 BEP, grease [°C]
    T_brg_warn_C:           float = 80.0
    T_brg_trip_C:           float = 100.0

    # Motor winding — Class F insulation [°C]
    T_motor_warn_C:         float = 130.0
    T_motor_trip_C:         float = 145.0

    # Vibration — ISO 10816-3, Group 1, rigid mounting [mm/s RMS]
    v_vib_warn_mm_s:        float = 4.5      # Zone B/C boundary
    v_vib_trip_mm_s:        float = 7.1      # Zone C/D boundary → trip

    # Aerodynamic
    PR_max:                 float = 1.25     # stage PR runaway trip
    P0_in_min_Pa:           float = 81000.0  # inlet pressure loss / blockage

    # Throttle [%]
    throttle_min_pct:       float = 30.0     # near-stall limit
    throttle_max_pct:       float = 95.0     # near-choke limit
    throttle_default_pct:   float = 85.0     # nominal operating point
    throttle_dwell_s:       float = 15.0     # settling time per sweep point

    def N_to_Hz(self, N_RPM: float) -> float:
        """Shaft speed [RPM] → VFD output frequency reference [Hz]."""
        return N_RPM / self.motor_nominal_RPM * self.motor_nominal_Hz

    def Hz_to_N(self, f_Hz: float) -> float:
        """VFD output frequency [Hz] → shaft speed [RPM]."""
        return f_Hz / self.motor_nominal_Hz * self.motor_nominal_RPM


# Instantiate — all cells below use this object
sp = RigSetpoints()

print(f"N_op          = {sp.N_op_RPM:.0f} RPM  →  VFD ref = {sp.N_to_Hz(sp.N_op_RPM):.2f} Hz")
print(f"N_sw_trip     = {sp.N_sw_trip_RPM:.0f} RPM  (110% of N_op)")
print(f"N_hw_trip     = {sp.N_hw_trip_RPM:.0f} RPM  (114% — hardwired EL1904 relay)")
print(f"Skip band 1   = {sp.skip1_lo_RPM:.0f}–{sp.skip1_hi_RPM:.0f} RPM  (≥{sp.skip_ramp_RPM_s:.0f} RPM/s)")
print(f"T_brg trip    = {sp.T_brg_trip_C:.0f} °C")
print(f"Vibration trip= {sp.v_vib_trip_mm_s:.1f} mm/s RMS (ISO 10816-3 Zone D)")
print(f"Throttle range= {sp.throttle_min_pct:.0f}–{sp.throttle_max_pct:.0f}%")

---
## 4. I/O List

Tags follow ISA-5.1 convention. Each instrument is mapped to its Beckhoff EtherCAT terminal.  
Safety-rated points use TwinSAFE terminals (EL1904 / EL2904).

In [ ]:
@dataclass
class AnalogInput:
    tag: str; description: str; unit: str
    lo: float; hi: float
    warn_lo: Optional[float]; warn_hi: Optional[float]
    trip_lo: Optional[float]; trip_hi: Optional[float]
    terminal: str = 'EL3174'; signal: str = '4-20 mA'

@dataclass
class DigitalInput:
    tag: str; description: str; normal_state: str
    safe: bool; terminal: str = 'EL1008'

@dataclass
class AnalogOutput:
    tag: str; description: str; unit: str
    lo: float; hi: float
    terminal: str = 'EL4004'; signal: str = '4-20 mA'

@dataclass
class DigitalOutput:
    tag: str; description: str; normal_state: str
    safe: bool; terminal: str = 'EL2008'


def build_io_list(sp: RigSetpoints) -> dict:
    AI = [
        AnalogInput('PT-101','Inlet total pressure P0_in — bellmouth pitot','Pa',
                    80000,110000,None,None,sp.P0_in_min_Pa,None),
        AnalogInput('PT-102','Inlet wall static — ISO 5801 measurement plane','Pa',
                    80000,110000,None,None,None,None),
        AnalogInput('PT-201','Stage exit total pressure P0_out','Pa',
                    90000,130000,None,None,None,sp.PR_max*101325),
        AnalogInput('PT-301','VFD actual output power (ACS880 CoE 0x6077)','kW',
                    0,400,None,330,None,360,terminal='ACS880-FECA01',signal='EtherCAT CoE'),
        AnalogInput('TT-101','Inlet total temperature T0_in','°C',0,80,None,50,None,60),
        AnalogInput('ZT-401','Throttle cone position (LVDT feedback)','%',0,100,None,None,None,None),
        AnalogInput('VT-201','Vibration — bearing A housing, radial','mm/s RMS',
                    0,20,sp.v_vib_warn_mm_s,None,None,sp.v_vib_trip_mm_s),
        AnalogInput('VT-202','Vibration — bearing B housing, radial','mm/s RMS',
                    0,20,sp.v_vib_warn_mm_s,None,None,sp.v_vib_trip_mm_s),
        AnalogInput('TE-201','Bearing A outer-ring temperature (drive end)','°C',
                    0,150,sp.T_brg_warn_C,None,None,sp.T_brg_trip_C,
                    terminal='EL3202',signal='PT100 3-wire'),
        AnalogInput('TE-202','Bearing B outer-ring temperature (non-drive end)','°C',
                    0,150,sp.T_brg_warn_C,None,None,sp.T_brg_trip_C,
                    terminal='EL3202',signal='PT100 3-wire'),
        AnalogInput('TE-301','Motor winding temperature (embedded Pt100)','°C',
                    0,200,sp.T_motor_warn_C,None,None,sp.T_motor_trip_C,
                    terminal='EL3202',signal='PT100 3-wire'),
        AnalogInput('ST-101','Shaft speed — OPR tachoprobe pulse','RPM',
                    0,5000,None,sp.N_sw_trip_RPM,None,sp.N_sw_trip_RPM,
                    terminal='EL1252',signal='24 V pulse, 1 µs'),
    ]
    DI = [
        DigitalInput('XS-001','E-stop — control room panel','NC',safe=True),
        DigitalInput('XS-002','E-stop — rig floor (local)','NC',safe=True),
        DigitalInput('XS-003','E-stop — laboratory entrance door','NC',safe=True),
        DigitalInput('XS-101','Overspeed relay — healthy (4 000 RPM hw trip)','NC',safe=True),
        DigitalInput('ZS-501','Test cell access door — closed & locked','NC',safe=True),
        DigitalInput('ZS-502','Rotor-end safety guard — in position','NC',safe=True),
        DigitalInput('MR-301','Motor protection relay — healthy','NC',safe=True),
        DigitalInput('STO-FB','ACS880 STO feedback — STO not active','NO',safe=True),
        DigitalInput('VFD-RDY','ACS880 ready-to-run','NO',safe=False),
        DigitalInput('VFD-RUN','ACS880 running confirmation','NO',safe=False),
        DigitalInput('VFD-FLT','ACS880 fault active','NC',safe=False),
        DigitalInput('HS-001','Supervisor key switch — RUN','NO',safe=False),
        DigitalInput('HS-101','Start pushbutton (illuminated)','NO',safe=False),
        DigitalInput('HS-102','Stop pushbutton (normal stop)','NC',safe=False),
    ]
    AO = [
        AnalogOutput('ZC-401','Throttle cone position setpoint','%',0,100),
        AnalogOutput('SC-301','ACS880 velocity demand (EtherCAT CoE 0x6042)','RPM',0,4200,
                     terminal='ACS880-FECA01',signal='CoE word 1 RPM/count'),
    ]
    DO = [
        DigitalOutput('XY-K1','Main contactor K1 — motor supply 400 V','NO',safe=True),
        DigitalOutput('XY-STO','ACS880 Safe Torque Off (STO) enable','NO',safe=True),
        DigitalOutput('HL-001','Red rotating beacon — rig running','NO',safe=False),
        DigitalOutput('HL-002','Amber beacon — alarm / warning active','NO',safe=False),
        DigitalOutput('AL-001','Audible alarm (horn, 95 dB)','NO',safe=False),
        DigitalOutput('TRG-001','DAQ trigger — test run start/end','NO',safe=False),
        DigitalOutput('TRG-002','DAQ trigger — operating point change','NO',safe=False),
    ]
    return {'AI': AI, 'DI': DI, 'AO': AO, 'DO': DO}


io = build_io_list(sp)

# ---- Display ----
print(f"Total I/O: {sum(len(io[k]) for k in io)} points")
print(f"  AI={len(io['AI'])}  DI={len(io['DI'])}  AO={len(io['AO'])}  DO={len(io['DO'])}")
print(f"  TwinSAFE: {sum(d.safe for d in io['DI'])} DI + {sum(d.safe for d in io['DO'])} DO")
print()
print(f"{'Tag':<10} {'Terminal':<14} {'Signal':<18} {'Warn Hi':>8} {'Trip Hi':>8}  Description")
print('─'*80)
for ai in io['AI']:
    wh = f"{ai.warn_hi:.0f}" if ai.warn_hi else '—'
    th = f"{ai.trip_hi:.0f}" if ai.trip_hi else '—'
    print(f"{ai.tag:<10} {ai.terminal:<14} {ai.signal:<18} {wh:>8} {th:>8}  {ai.description[:40]}")

In [ ]:
print(f"{'Tag':<10} {'Safe':>5}  {'Normal':>7}  Description")
print('─'*65)
for di in io['DI']:
    sf = 'SIL-2' if di.safe else '     '
    print(f"{di.tag:<10} {sf:>5}  {di.normal_state:>7}  {di.description}")
print()
for ao in io['AO']:
    print(f"{ao.tag:<10} AO  {ao.terminal:<16} {ao.signal}  {ao.description}")
for do_ in io['DO']:
    sf = '[SIL-2]' if do_.safe else '       '
    print(f"{do_.tag:<10} DO  {sf}  {do_.description}")

---
## 5. State Machine

The rig follows a **9-state sequential machine** enforced in TwinCAT 3 as a  
`FUNCTION_BLOCK FB_RigStateMachine` written in Structured Text (ST) / Sequential  
Function Chart (SFC). Transitions are validated against the permitted transition table —  
no illegal state jumps are possible.

```
IDLE → PRE_START → SPIN_UP → HOLD_IDLE → AT_SPEED ⇄ SWEEP
                                                  ↓
                                            COAST_DOWN → IDLE
                                  (any state) → TRIP → FAULT → IDLE
```

State is held in `RETAIN` persistent memory so a power cycle restores the last known state.

In [ ]:
class RigState(Enum):
    IDLE        = auto()   # Rotor stopped, all outputs de-energised
    PRE_START   = auto()   # Checklist running, awaiting start command
    SPIN_UP     = auto()   # ACS880 ramping, skip bands traversed
    HOLD_IDLE   = auto()   # At N_min, bearing warm-up dwell
    AT_SPEED    = auto()   # At N_op, ready for test
    SWEEP       = auto()   # Throttle sweep in progress
    ALARM       = auto()   # Warning active, running degraded
    COAST_DOWN  = auto()   # Normal stop: ACS880 ramping to zero
    TRIP        = auto()   # Safety: STO asserted immediately
    FAULT       = auto()   # Post-trip: locked out, manual reset required


class RigStateMachine:
    """
    TwinCAT 3 implementation:
      FB_RigStateMachine FUNCTION_BLOCK (IEC 61131-3 ST)
      Called from CTRL_TASK every 5 ms.
      Transitions logged via TwinCAT EventLogger.
    """
    TRANSITIONS: dict = {
        RigState.IDLE:       [RigState.PRE_START, RigState.FAULT],
        RigState.PRE_START:  [RigState.SPIN_UP, RigState.IDLE, RigState.FAULT],
        RigState.SPIN_UP:    [RigState.HOLD_IDLE, RigState.TRIP, RigState.FAULT],
        RigState.HOLD_IDLE:  [RigState.AT_SPEED, RigState.TRIP, RigState.FAULT],
        RigState.AT_SPEED:   [RigState.SWEEP, RigState.ALARM,
                              RigState.COAST_DOWN, RigState.TRIP],
        RigState.SWEEP:      [RigState.AT_SPEED, RigState.ALARM,
                              RigState.COAST_DOWN, RigState.TRIP],
        RigState.ALARM:      [RigState.AT_SPEED, RigState.COAST_DOWN, RigState.TRIP],
        RigState.COAST_DOWN: [RigState.IDLE, RigState.TRIP],
        RigState.TRIP:       [RigState.FAULT],
        RigState.FAULT:      [RigState.IDLE],
    }

    def __init__(self, sp: RigSetpoints):
        self.sp         = sp
        self.state      = RigState.IDLE
        self.t_enter    = time.monotonic()
        self.history    = [(RigState.IDLE, 0.0, 'init')]
        self.trip_log   = []

    def transition(self, new_state: RigState, reason: str = '') -> bool:
        if new_state not in self.TRANSITIONS.get(self.state, []):
            return False
        self.state   = new_state
        self.t_enter = time.monotonic()
        self.history.append((new_state, self.t_enter, reason))
        if new_state == RigState.TRIP:
            self.trip_log.append(reason)
        return True

    @property
    def t_in_state(self) -> float:
        return time.monotonic() - self.t_enter

    def in_skip_band(self, N: float) -> bool:
        return self.sp.skip1_lo_RPM <= N <= self.sp.skip1_hi_RPM


# Walk through a normal run sequence
sm = RigStateMachine(sp)
sequence = [
    (RigState.PRE_START, 'operator pressed Start, checklist passed'),
    (RigState.SPIN_UP,   'all checks green'),
    (RigState.HOLD_IDLE, f'reached {sp.N_min_stable_RPM:.0f} RPM'),
    (RigState.AT_SPEED,  f'warm-up complete, at N_op={sp.N_op_RPM:.0f} RPM'),
    (RigState.SWEEP,     'operator initiated φ-sweep'),
    (RigState.AT_SPEED,  'sweep complete'),
    (RigState.COAST_DOWN,'operator pressed Stop'),
    (RigState.IDLE,      'rotor at standstill'),
]
print(f"{'Step':<4} {'Result':<6} {'State':<14}  Reason")
print('─'*65)
for i,(state,reason) in enumerate(sequence, 1):
    ok = sm.transition(state, reason)
    print(f"{i:<4} {'OK' if ok else 'DENIED':<6} {sm.state.name:<14}  {reason}")

# Verify illegal transition is blocked
print()
print("Illegal transition test (IDLE → AT_SPEED directly):")
sm2 = RigStateMachine(sp)
result = sm2.transition(RigState.AT_SPEED, 'bypass attempt')
print(f"  Result: {'BLOCKED ✓' if not result else 'ALLOWED — BUG!'}  (sm still in {sm2.state.name})")

---
## 6. Interlock Logic

Three layers of protection, independent of each other:

| Layer | Implementation | Response time | Mechanism |
|---|---|---|---|
| 1 — Hardwired | E-stop relay chain, overspeed relay | < 20 ms | Cuts K1 contactor directly |
| 2 — TwinSAFE | EL6910 safety CPU, EL1904/EL2904 | 10 ms task | Asserts ACS880 STO + opens K1 |
| 3 — Software | CTRL_TASK evaluate_interlocks() | 5 ms task | Controlled coast-down |

The Python function below mirrors the TwinSAFE FBD logic implemented with  
`SF_EmergencyStop`, `SF_GuardDoor`, `SF_ESPE`, and `SF_STO` certified library blocks.

In [ ]:
@dataclass
class ProcessValues:
    """
    Live process value snapshot — filled by MOTION_TASK I/O scan every 1 ms.
    Declared in GVL_PV (global variable list, non-retain).
    """
    N_RPM:           float = 0.0
    T_brg_A_C:       float = 25.0
    T_brg_B_C:       float = 25.0
    T_motor_C:       float = 25.0
    V_vib_A_mm_s:    float = 0.0
    V_vib_B_mm_s:    float = 0.0
    P0_in_Pa:        float = 101325.0
    P0_out_Pa:       float = 101325.0
    T0_in_C:         float = 15.0
    throttle_pct:    float = 85.0
    motor_kW:        float = 0.0
    # Safety DIs (EL1904 — NC = True when healthy)
    estop_ok:        bool  = True
    overspeed_ok:    bool  = True
    door_locked:     bool  = True
    guard_ok:        bool  = True
    motor_relay_ok:  bool  = True
    sto_fb_ok:       bool  = True
    # Standard DIs (EL1008)
    vfd_ready:       bool  = False
    vfd_running:     bool  = False
    vfd_fault:       bool  = False
    key_run:         bool  = False


@dataclass
class InterlockResult:
    trip:          bool
    alarm:         bool
    trip_reasons:  list = field(default_factory=list)
    alarm_reasons: list = field(default_factory=list)


def evaluate_interlocks(pv: ProcessValues, sp: RigSetpoints) -> InterlockResult:
    """
    Full interlock evaluation.  Mirrors TwinSAFE FBD logic.
    Trip → STO asserted + K1 opens.  Alarm → logged, operator notified.
    """
    trips, alarms = [], []

    # Layer 1 — hardwired (logged here for HMI; actual cut is relay chain)
    if not pv.estop_ok:       trips.append('E-STOP activated — XS-001/002/003 chain open')
    if not pv.overspeed_ok:   trips.append('Hardwired overspeed relay XS-101 tripped')
    if not pv.motor_relay_ok: trips.append('Motor protection relay MR-301 tripped')

    # Layer 2 — TwinSAFE software trips
    if pv.N_RPM >= sp.N_sw_trip_RPM:
        trips.append(f'Overspeed: {pv.N_RPM:.0f} ≥ {sp.N_sw_trip_RPM:.0f} RPM')
    if pv.T_brg_A_C >= sp.T_brg_trip_C:
        trips.append(f'Bearing A over-temp: {pv.T_brg_A_C:.1f}°C ≥ {sp.T_brg_trip_C:.0f}°C')
    if pv.T_brg_B_C >= sp.T_brg_trip_C:
        trips.append(f'Bearing B over-temp: {pv.T_brg_B_C:.1f}°C ≥ {sp.T_brg_trip_C:.0f}°C')
    if pv.T_motor_C >= sp.T_motor_trip_C:
        trips.append(f'Motor over-temp: {pv.T_motor_C:.1f}°C ≥ {sp.T_motor_trip_C:.0f}°C')
    if pv.V_vib_A_mm_s >= sp.v_vib_trip_mm_s:
        trips.append(f'Vibration A (ISO Zone D): {pv.V_vib_A_mm_s:.1f} mm/s')
    if pv.V_vib_B_mm_s >= sp.v_vib_trip_mm_s:
        trips.append(f'Vibration B (ISO Zone D): {pv.V_vib_B_mm_s:.1f} mm/s')
    if not pv.door_locked and pv.N_RPM > sp.N_min_stable_RPM:
        trips.append('Access door ZS-501 opened while rig running')
    if not pv.guard_ok and pv.N_RPM > sp.N_min_stable_RPM:
        trips.append('Safety guard ZS-502 removed while rig running')
    if pv.vfd_fault:
        trips.append('ACS880 drive fault — VFD-FLT active')
    PR = pv.P0_out_Pa / max(pv.P0_in_Pa, 1.0)
    if PR >= sp.PR_max:
        trips.append(f'Stage PR {PR:.3f} ≥ trip limit {sp.PR_max:.2f}')
    if pv.P0_in_Pa <= sp.P0_in_min_Pa and pv.N_RPM > sp.N_min_stable_RPM:
        trips.append(f'Inlet pressure loss: {pv.P0_in_Pa:.0f} Pa ≤ {sp.P0_in_min_Pa:.0f} Pa')

    # Layer 3 — software alarms
    if pv.T_brg_A_C >= sp.T_brg_warn_C: alarms.append(f'Bearing A {pv.T_brg_A_C:.1f}°C (ISO Zone C)')
    if pv.T_brg_B_C >= sp.T_brg_warn_C: alarms.append(f'Bearing B {pv.T_brg_B_C:.1f}°C (ISO Zone C)')
    if pv.T_motor_C >= sp.T_motor_warn_C: alarms.append(f'Motor winding {pv.T_motor_C:.1f}°C')
    if pv.V_vib_A_mm_s >= sp.v_vib_warn_mm_s: alarms.append(f'Vibration A {pv.V_vib_A_mm_s:.1f} mm/s (Zone B/C)')
    if pv.V_vib_B_mm_s >= sp.v_vib_warn_mm_s: alarms.append(f'Vibration B {pv.V_vib_B_mm_s:.1f} mm/s (Zone B/C)')
    if pv.N_RPM > sp.N_op_RPM * 1.05: alarms.append(f'Speed {pv.N_RPM:.0f} RPM > 105% of N_op')
    if pv.motor_kW > 330: alarms.append(f'Motor power {pv.motor_kW:.0f} kW > 330 kW (103%)')

    return InterlockResult(trip=len(trips)>0, alarm=len(alarms)>0,
                           trip_reasons=trips, alarm_reasons=alarms)


print("Interlock function defined — see test scenarios in the next cell")

In [ ]:
# ---- Test scenarios ----
scenarios = [
    ("Normal running at design point",
     ProcessValues(N_RPM=3500, T_brg_A_C=55, T_brg_B_C=52, T_motor_C=85,
                   V_vib_A_mm_s=1.2, V_vib_B_mm_s=1.0,
                   P0_in_Pa=101325, P0_out_Pa=111450,
                   estop_ok=True, overspeed_ok=True, door_locked=True,
                   guard_ok=True, motor_relay_ok=True, sto_fb_ok=True)),
    ("Bearing A approaching alarm threshold (78°C)",
     ProcessValues(N_RPM=3500, T_brg_A_C=78, T_brg_B_C=50,
                   P0_in_Pa=101325, P0_out_Pa=111450,
                   estop_ok=True, overspeed_ok=True, door_locked=True,
                   guard_ok=True, motor_relay_ok=True, sto_fb_ok=True)),
    ("Bearing A at trip threshold (105°C)",
     ProcessValues(N_RPM=3500, T_brg_A_C=105, T_brg_B_C=50,
                   P0_in_Pa=101325, P0_out_Pa=111450,
                   estop_ok=True, overspeed_ok=True, door_locked=True,
                   guard_ok=True, motor_relay_ok=True, sto_fb_ok=True)),
    ("E-stop activated",
     ProcessValues(N_RPM=3500, T_brg_A_C=55, T_brg_B_C=52,
                   P0_in_Pa=101325, P0_out_Pa=111450,
                   estop_ok=False, overspeed_ok=True, door_locked=True,
                   guard_ok=True, motor_relay_ok=True, sto_fb_ok=True)),
    ("Overspeed — software trip (3900 RPM)",
     ProcessValues(N_RPM=3900, T_brg_A_C=55, T_brg_B_C=52,
                   P0_in_Pa=101325, P0_out_Pa=111450,
                   estop_ok=True, overspeed_ok=True, door_locked=True,
                   guard_ok=True, motor_relay_ok=True, sto_fb_ok=True)),
]

for name, pv in scenarios:
    r = evaluate_interlocks(pv, sp)
    status = '🔴 TRIP' if r.trip else ('🟡 ALARM' if r.alarm else '🟢 OK')
    print(f"\n{status}  {name}")
    for t in r.trip_reasons:  print(f"   ✗ {t}")
    for a in r.alarm_reasons: print(f"   ⚠ {a}")

---
## 7. Pre-Start Checklist

All 14 items must pass before the state machine allows `IDLE → PRE_START`. In TwinCAT this runs as `FB_PreStartCheck` called once when the operator presses the illuminated Start pushbutton (HS-101). Results appear on the HMI checklist page with green/red indicators per item.

In [ ]:
def pre_start_check(pv: ProcessValues, sp: RigSetpoints) -> tuple:
    fails = []
    if not pv.estop_ok:       fails.append('E-stop chain not healthy — check XS-001, XS-002, XS-003')
    if not pv.overspeed_ok:   fails.append('Overspeed relay XS-101 not healthy')
    if not pv.door_locked:    fails.append('Test cell door ZS-501 not locked')
    if not pv.guard_ok:       fails.append('Rotor-end guard ZS-502 not confirmed in place')
    if not pv.motor_relay_ok: fails.append('Motor protection relay MR-301 not healthy')
    if pv.vfd_fault:          fails.append('ACS880 fault active — reset drive before starting')
    if not pv.vfd_ready:      fails.append('ACS880 not ready — check DC bus charge and drive enable')
    if not pv.sto_fb_ok:      fails.append('ACS880 STO feedback: STO unexpectedly active')
    if not pv.key_run:        fails.append('Supervisor key switch HS-001 not in RUN position')
    if pv.N_RPM > 50:         fails.append(f'Rotor not at standstill: {pv.N_RPM:.0f} RPM')
    if pv.T_brg_A_C > sp.T_brg_warn_C: fails.append(f'Bearing A already high ({pv.T_brg_A_C:.0f}°C)')
    if pv.T_brg_B_C > sp.T_brg_warn_C: fails.append(f'Bearing B already high ({pv.T_brg_B_C:.0f}°C)')
    if pv.T_motor_C > sp.T_motor_warn_C: fails.append(f'Motor temp already high ({pv.T_motor_C:.0f}°C)')
    if abs(pv.P0_in_Pa - 101325) > 5000: fails.append(f'PT-101 abnormal ({pv.P0_in_Pa:.0f} Pa)')
    return (len(fails) == 0, fails)


# All healthy
pv_healthy = ProcessValues(
    estop_ok=True, overspeed_ok=True, door_locked=True, guard_ok=True,
    motor_relay_ok=True, sto_fb_ok=True, vfd_fault=False, vfd_ready=True,
    key_run=True, N_RPM=0, T_brg_A_C=22, T_brg_B_C=22,
    T_motor_C=28, P0_in_Pa=101325,
)
ok, fails = pre_start_check(pv_healthy, sp)
print("Pre-start check — all healthy:")
print(f"  Result: {'✅ ALL OK — cleared for start' if ok else '❌ FAILED'}")

# Simulate a failed check
pv_bad = ProcessValues(
    estop_ok=True, overspeed_ok=True, door_locked=False,   # door not locked
    guard_ok=True, motor_relay_ok=True, sto_fb_ok=True,
    vfd_fault=False, vfd_ready=True, key_run=False,         # key switch off
    N_RPM=0, T_brg_A_C=22, T_brg_B_C=22, T_motor_C=28, P0_in_Pa=101325,
)
ok2, fails2 = pre_start_check(pv_bad, sp)
print("\nPre-start check — two failures:")
print(f"  Result: {'✅ OK' if ok2 else '❌ FAILED'}")
for f in fails2:
    print(f"  ✗ {f}")

---
## 8. Speed Ramp Controller & Skip Band Simulation

The ACS880 receives a velocity demand each EtherCAT cycle via CoE object 0x6042 (CSV mode).  
`SpeedController.step()` runs in `MOTION_TASK` every 1 ms and manages the ramp profile.

The **skip band** (2 191–2 591 RPM) must be traversed in < 5 s to avoid resonance dwell.  
Normal ramp rate is 58 RPM/s (3 500 RPM / 60 s). Through the skip band the ramp boosts  
to ≥ 80 RPM/s, traversing the 400 RPM band in ≤ 5 s.

In [ ]:
class SpeedController:
    """
    ACS880 CSV velocity demand management.
    TwinCAT: FB_SpeedControl, MOTION_TASK, 1 ms cycle.
    """
    def __init__(self, sp: RigSetpoints):
        self.sp      = sp
        self.N_ref   = 0.0    # current velocity demand [RPM]
        self.N_target = 0.0

    def set_target(self, N_RPM: float):
        self.N_target = max(0.0, min(float(N_RPM), self.sp.N_op_RPM))

    def step(self, N_actual: float, dt_s: float) -> float:
        ramp = self.sp.N_op_RPM / self.sp.ramp_up_s   # ≈ 58 RPM/s
        if self.sp.skip1_lo_RPM <= N_actual <= self.sp.skip1_hi_RPM:
            ramp = max(ramp, self.sp.skip_ramp_RPM_s)
        delta = self.N_target - self.N_ref
        step  = ramp * dt_s
        self.N_ref += step * np.sign(delta) if abs(delta) > step else delta
        self.N_ref = float(np.clip(self.N_ref, 0, self.sp.N_op_RPM))
        return self.N_ref

    @property
    def f_ref_Hz(self) -> float:
        return self.sp.N_to_Hz(self.N_ref)


# Simulate run-up from 0 to N_op (dt = 100 ms for speed)
sc = SpeedController(sp)
sc.set_target(sp.N_op_RPM)
dt = 0.100          # 100 ms simulation step
t_sim, N_profile, ramp_rate = [0.0], [0.0], [0.0]
N_sim = 0.0
for _ in range(800):
    t_sim.append(t_sim[-1] + dt)
    N_new = sc.step(N_sim, dt)
    ramp_rate.append((N_new - N_sim) / dt)
    N_sim = N_new
    N_profile.append(N_sim)

t_arr = np.array(t_sim); N_arr = np.array(N_profile)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
ax1.plot(t_arr, N_arr, color='#185FA5', lw=2)
ax1.axhspan(sp.skip1_lo_RPM, sp.skip1_hi_RPM, alpha=0.15, color='#E24B4A', label='Skip band')
ax1.axhline(sp.N_op_RPM, color='#1D9E75', lw=1.2, ls='--', label=f'N_op {sp.N_op_RPM:.0f} RPM')
ax1.axhline(sp.N_min_stable_RPM, color='#BA7517', lw=1.0, ls=':', label=f'N_min_stable {sp.N_min_stable_RPM:.0f} RPM')
ax1.set_ylabel('Shaft speed [RPM]')
ax1.legend(fontsize=9); ax1.grid(alpha=0.3)
ax1.set_title('ACS880 velocity demand — run-up ramp with skip-band boost', fontsize=11)

ax2.plot(t_arr[1:], ramp_rate[1:], color='#D85A30', lw=1.5)
ax2.axhline(sp.skip_ramp_RPM_s, color='#E24B4A', ls='--', lw=1, label=f'Min skip rate {sp.skip_ramp_RPM_s:.0f} RPM/s')
ax2.set_ylabel('Ramp rate [RPM/s]'); ax2.set_xlabel('Time [s]')
ax2.legend(fontsize=9); ax2.grid(alpha=0.3)

# Annotate skip band traversal time
mask = (N_arr >= sp.skip1_lo_RPM) & (N_arr <= sp.skip1_hi_RPM)
if mask.any():
    t_traverse = t_arr[mask][-1] - t_arr[mask][0]
    ax1.annotate(f'Skip band\n{t_traverse:.1f} s < 5 s ✓',
                 xy=(t_arr[mask][len(t_arr[mask])//2], sp.skip1_lo_RPM + 200),
                 fontsize=9, color='#A32D2D',
                 arrowprops=dict(arrowstyle='->', color='#A32D2D'),
                 xytext=(t_arr[mask][len(t_arr[mask])//2] + 3, sp.skip1_hi_RPM + 300))

plt.tight_layout()
plt.savefig('/mnt/user-data/outputs/speed_ramp.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Skip band traversal time: {t_traverse:.2f} s  (limit: < 5 s)  {'✓' if t_traverse < 5 else '✗'}")

---
## 9. Throttle Sweep Controller

The EL4004 channel ZC-401 drives a 4-20 mA proportional actuator on the downstream throttle cone.  
Closing the throttle raises back-pressure and reduces φ toward surge.

**16-point sweep schedule** matches the Tian et al. (2018) test matrix:  
φ = 0.172 (near choke) down to φ = 0.084 (near stall).

Each step: throttle moves → `TRG-002` pulse to DAQ (marks operating point change)  
→ `throttle_dwell_s` settling → `TRG-001` rising edge → DAQ capture window.

In [ ]:
class ThrottleController:
    """
    Throttle cone position management.
    TwinCAT: FB_ThrottleControl, CTRL_TASK, 5 ms cycle.
    Output ZC-401 written to EL4004 AO channel.
    """
    def __init__(self, sp: RigSetpoints):
        self.sp       = sp
        self.position = sp.throttle_default_pct
        self.target   = sp.throttle_default_pct
        self._sweep   = []
        self._idx     = 0

    def set_position(self, pct: float):
        self.target = float(np.clip(pct, self.sp.throttle_min_pct, self.sp.throttle_max_pct))

    def open_default(self):
        self.set_position(self.sp.throttle_default_pct)

    def generate_sweep(self, n_points: int = 16, direction: str = 'closing') -> list:
        pts = np.linspace(self.sp.throttle_max_pct, self.sp.throttle_min_pct, n_points).tolist()
        if direction == 'opening':
            pts = list(reversed(pts))
        self._sweep = pts; self._idx = 0
        return pts

    def advance(self) -> tuple:
        """Move to next sweep point. Returns (complete, position)"""
        if self._idx >= len(self._sweep):
            return True, self.position
        self.set_position(self._sweep[self._idx])
        self.position = self.target
        self._idx += 1
        return self._idx >= len(self._sweep), self.target


tc = ThrottleController(sp)
pts = tc.generate_sweep(16, 'closing')

# Build schedule table
print(f"{'Pt':>3}  {'Throttle':>10}  {'mA out':>8}  {'TRG-002':>8}  {'Dwell [s]':>10}  {'TRG-001':>8}")
print('─'*60)
t_cumul = 0.0
for i, pct in enumerate(pts, 1):
    mA = 4 + (pct / 100) * 16           # 4-20 mA scaling
    t_cumul += sp.throttle_dwell_s
    print(f"{i:>3}  {pct:>9.1f}%  {mA:>7.2f} mA  {'pulse':>8}  {sp.throttle_dwell_s:>10.0f}  {'↑':>8}")

print(f"\nTotal sweep duration estimate: {t_cumul:.0f} s ({t_cumul/60:.1f} min) + actuator travel")
print(f"  ({len(pts)} points × {sp.throttle_dwell_s:.0f} s dwell)")

In [ ]:
# Plot the sweep schedule
fig, ax = plt.subplots(figsize=(11, 4))
t_steps = np.arange(len(pts)) * sp.throttle_dwell_s
ax.step(t_steps, pts, where='post', color='#185FA5', lw=2, label='Throttle position')
ax.fill_between(t_steps, pts, step='post', alpha=0.12, color='#185FA5')
for i, (t, p) in enumerate(zip(t_steps, pts)):
    ax.axvline(t, color='#E24B4A', lw=0.8, ls=':', alpha=0.6)
    ax.annotate(f'{i+1}', xy=(t + sp.throttle_dwell_s*0.1, p - 2.5), fontsize=7.5, color='#993C1D')
ax.set_xlabel('Elapsed time [s]')
ax.set_ylabel('Throttle position [%]')
ax.set_title(f'16-point φ-sweep schedule  ({sp.throttle_dwell_s:.0f} s dwell per point,  choke → stall)', fontsize=11)
ax.annotate('TRG-002 pulse\n(DAQ op-point marker)', xy=(0, pts[0]), xytext=(15, pts[0]+8),
            fontsize=8, color='#E24B4A',
            arrowprops=dict(arrowstyle='->', color='#E24B4A'))
ax.annotate('TRG-001 pulse\n(DAQ capture trigger)', xy=(sp.throttle_dwell_s*0.9, pts[0]-1),
            xytext=(sp.throttle_dwell_s*0.9 + 15, pts[0]-10),
            fontsize=8, color='#1D9E75',
            arrowprops=dict(arrowstyle='->', color='#1D9E75'))
ax.set_xlim(-2, t_steps[-1] + sp.throttle_dwell_s + 5)
ax.grid(alpha=0.3); ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('/mnt/user-data/outputs/throttle_sweep.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 10. TwinCAT ADS — Python Research Interface

The CX2040 runs an ADS server accessible from any PC on the lab network.  
Your students can read and write PLC variables in real time using the `pyads` library  
without touching the TwinCAT project or the safety layer.

```bash
pip install pyads
```

The code below shows the typical usage pattern. In the lab, replace  
`'192.168.1.10.1.1'` with the actual AMS Net ID assigned at commissioning.

In [ ]:
ADS_EXAMPLE = '''
import pyads
import numpy as np
import time

# Connect to CX2040 ADS server
AMS_NET_ID = '192.168.1.10.1.1'   # set at TwinCAT commissioning
plc = pyads.Connection(AMS_NET_ID, pyads.PORT_TC3PLC1)
plc.open()

# ── Read process values ────────────────────────────────────────
N       = plc.read_by_name('GVL_PV.N_RPM',          pyads.PLCTYPE_LREAL)
T_brg_A = plc.read_by_name('GVL_PV.T_brg_A_C',      pyads.PLCTYPE_LREAL)
P0_in   = plc.read_by_name('GVL_PV.P0_in_Pa',       pyads.PLCTYPE_LREAL)
state   = plc.read_by_name('GVL_PV.rig_state',      pyads.PLCTYPE_INT)

print(f"Speed: {N:.1f} RPM   Bearing A: {T_brg_A:.1f}°C   P0_in: {P0_in:.0f} Pa")

# ── Write throttle setpoint ────────────────────────────────────
plc.write_by_name('GVL_PV.throttle_target_pct', 65.0, pyads.PLCTYPE_LREAL)

# ── Trigger a φ-sweep ─────────────────────────────────────────
plc.write_by_name('GVL_CMD.cmd_start_sweep', True, pyads.PLCTYPE_BOOL)

# ── Stream data at 100 ms intervals ───────────────────────────
log = []
for _ in range(100):
    N_live = plc.read_by_name('GVL_PV.N_RPM', pyads.PLCTYPE_LREAL)
    log.append(N_live)
    time.sleep(0.1)

# ── OPC-UA alternative (for NI DAQ / LabVIEW) ─────────────────
# Server URL: opc.tcp://192.168.1.10:4840
# Node ID format: ns=4;s=GVL_PV.N_RPM

plc.close()
'''

print("ADS interface code (execute in lab with CX2040 connected):")
print(ADS_EXAMPLE)

---
## 11. Full System Summary

Run the cell below to print the complete design summary — hardware, I/O count, setpoints, and state machine transitions.

In [ ]:
SEP = '─' * 64

print(f"{'='*64}")
print(f"  PLC CONTROL SYSTEM  —  BECKHOFF CX2040 / TWINCAT 3")
print(f"{'='*64}")

hw = [
    ('Controller',   'Beckhoff CX2040 — Intel Core i7, 4 GB RAM, 8 GB CFast'),
    ('OS / runtime', 'Windows 10 IoT + TwinCAT 3 (dedicated RT cores)'),
    ('Safety CPU',   'EL6910 TwinSAFE — SIL 2 / PL d'),
    ('Fieldbus',     'EtherCAT (IEC 61158), 125 µs bus cycle'),
    ('VFD',          'ABB ACS880-01-385A-3, 355 kW, 400 V'),
    ('VFD link',     'FECA-01 EtherCAT adapter, CiA 402 CSV, 1 ms update'),
    ('VFD STO',      'IEC 61800-5-2 STO via EL2904 TwinSAFE DO'),
    ('HMI',          'Beckhoff CP2916, 15.6″ multi-touch (IP65)'),
    ('HMI software', 'TwinCAT HMI TE2000 — HTML5, browser-accessible'),
    ('Research I/F', 'pyads (ADS) + OPC-UA TF6100'),
]
print(f"\nHARDWARE")
print(SEP)
for k,v in hw: print(f"  {k:<16}: {v}")

print(f"\nSETTINGS")
print(SEP)
cfg = [
    ('N_op',          f'{sp.N_op_RPM:.0f} RPM  →  {sp.N_to_Hz(sp.N_op_RPM):.1f} Hz VFD reference'),
    ('N_sw_trip',     f'{sp.N_sw_trip_RPM:.0f} RPM  (110% — TwinSAFE)'),
    ('N_hw_trip',     f'{sp.N_hw_trip_RPM:.0f} RPM  (114% — EL1904 relay)'),
    ('Skip band 1',   f'{sp.skip1_lo_RPM:.0f}–{sp.skip1_hi_RPM:.0f} RPM  (≥{sp.skip_ramp_RPM_s:.0f} RPM/s)'),
    ('Ramp up/down',  f'{sp.ramp_up_s:.0f} s / {sp.ramp_dn_s:.0f} s'),
    ('Bearing warmup',f'{sp.bearing_warmup_s:.0f} s at {sp.N_min_stable_RPM:.0f} RPM'),
    ('T_bearing',     f'warn {sp.T_brg_warn_C:.0f}°C  /  trip {sp.T_brg_trip_C:.0f}°C'),
    ('T_motor',       f'warn {sp.T_motor_warn_C:.0f}°C  /  trip {sp.T_motor_trip_C:.0f}°C'),
    ('Vibration',     f'warn {sp.v_vib_warn_mm_s:.1f} mm/s  /  trip {sp.v_vib_trip_mm_s:.1f} mm/s (ISO 10816-3)'),
    ('Throttle',      f'{sp.throttle_min_pct:.0f}–{sp.throttle_max_pct:.0f}%  (default {sp.throttle_default_pct:.0f}%)'),
    ('Sweep dwell',   f'{sp.throttle_dwell_s:.0f} s per φ point'),
]
for k,v in cfg: print(f"  {k:<18}: {v}")

print(f"\nI/O  ({sum(len(io[k]) for k in io)} points total)")
print(SEP)
print(f"  AI {len(io['AI'])}  DI {len(io['DI'])}  AO {len(io['AO'])}  DO {len(io['DO'])}")
print(f"  TwinSAFE: {sum(d.safe for d in io['DI'])} DI (EL1904)  +  {sum(d.safe for d in io['DO'])} DO (EL2904)")
print(f"\n{'='*64}")